In [141]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_1_1.csv')
# print(df.info())
# print(df.head())
df['합계'] = df['금액1'] + df['금액2']
df1 = df.groupby(['year', 'gender', '지역코드'])['합계'].sum().reset_index().groupby(['year','지역코드', 'gender'])['합계'].sum().reset_index()
df2 = df1.pivot(index=['year', '지역코드'], columns='gender', values='합계')
df2 = df2.fillna(0)
df2['차이'] = np.abs(df2[0] - df2[1])
df2['차이'].sort_values(ascending=False)

year  지역코드
2018  143     168869.0
2015  195     161121.0
2016  157     137959.0
2018  100     131165.0
      116     130947.0
                ...   
2019  137      27974.0
2016  161      25300.0
2018  174      22578.0
      151      21695.0
2016  189      16681.0
Name: 차이, Length: 91, dtype: float64

In [142]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_1_2.csv')
# print(df.info())
# print(df.head())

df1 = df[df['구분'] == '발생건수'].set_index('연도').drop(columns=['구분'])
df2 = df[df['구분'] == '검거건수'].set_index('연도').drop(columns=['구분'])
df3 = df2 / df1
df3

def check(x):
    return x == x.max()
df4 = df3.apply(check)
# df3
df4
print(df2[df4].fillna(0).sum().sum().round(0))

36977.0


In [55]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_1_3.csv')
# print(df.info())
# print(df.head())
df['평균만족도'] = df['평균만족도'].fillna(df['평균만족도'].mean())
df['근속연수'] = df['근속연수'].fillna(np.floor(df.groupby(['부서', '등급'])['근속연수'].transform('mean')))
# df.info()
a = df[(df['부서'] == 'HR') & (df['등급'] == 'A')]['근속연수'].mean()
b = df[(df['부서'] == 'Sales') & (df['등급'] == 'B')]['교육참가횟수'].mean()
print(a + b)

25.225


In [115]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_2_test.csv')
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import f1_score, get_scorer_names
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
# f1_macro

train_y = train_df['라벨']
train_X = train_df.drop(['ID', '라벨'],axis=1)
test_y = test_df['라벨']
test_X = test_df.drop(['ID', '라벨'],axis=1)

train_X, valid_X, train_y, valid_y = train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

obj = train_X.select_dtypes('object').columns.tolist()
num = train_X.select_dtypes('number').columns.tolist()
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_X_encoder = encoder.fit_transform(train_X[obj])
valid_X_encoder = encoder.transform(valid_X[obj])
test_X_encoder = encoder.transform(test_X[obj])

train_X = np.concatenate([train_X_encoder, train_X[num]], axis=1)
valid_X = np.concatenate([valid_X_encoder, valid_X[num]], axis=1)
test_X = np.concatenate([test_X_encoder, test_X[num]], axis=1)

rf=RandomForestClassifier()
rf.fit(train_X, train_y)
pred1 = rf.predict(valid_X)
# print(help(f1_score))
print(f1_score(pred1, valid_y, average='macro'))
# pred2 = rf.predict(test_X)
# pd.DataFrame(pred2, columns=['pred']).to_csv('result.csv', index=False)

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv=GridSearchCV(estimator=rf, param_grid=params, scoring='f1_macro', cv=3)
# print(help(GridSearchCV))
cv.fit(train_X, train_y)
pred3 = cv.predict(valid_X)
print(f1_score(pred3, valid_y, average='macro'))
pred4 = cv.predict(test_X)

0.30570505920344454
0.3049262855088098


In [100]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_3_1.csv')
# print(df.info())
# print(df.head())

train_df = df.iloc[:140,:]
train_df=train_df.drop(columns=['id'])
test_df = df.iloc[140:,:]

import statsmodels.api as sm
import statsmodels.formula.api as smf
model = smf.ols('design ~ tenure + f2 + f3 + f4 + f5', data=train_df).fit()
print((model.pvalues[1:] > 0.05).sum())
train_y=train_df['design']
train_X=train_df.drop(columns=['design'])
pred1 = model.predict(train_X)
from scipy.stats import pearsonr
print(pearsonr(pred1, train_y).statistic.round(2))
test_y=test_df['design']
test_X=test_df.drop(columns=['id', 'design'])
pred2 = model.predict(test_X)
from sklearn.metrics import root_mean_squared_error
print(np.round(root_mean_squared_error(pred2,test_y),0))

2
0.91
4.0


In [101]:
# 3-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/9_3_2.csv')
# print(df.info())
# print(df.head())
import statsmodels.api as sm
import statsmodels.formula.api as smf
model = smf.glm('churn ~ col1 + col2 + Phone_Service + Tech_Insurance', data=df, family=sm.families.Binomial()).fit()
print(model.pvalues['col1'].round(3))
print(np.exp(model.params['Phone_Service']).round(3))
pred1 = model.predict(df)
print((pred1 >= 0.3).sum())

0.0
1.867
450
